# Clase 138 — 1D CNNs y WaveNet

`Conv1D` es la alternativa **paralelizable** a las RNN para secuencias. Con
**convoluciones causales dilatadas** (WaveNet, 2016) se logra un **receptive field**
enorme con pocas capas: `dilation_rate` 1, 2, 4, 8, ... crece exponencialmente.

Requiere: `tensorflow` / `keras` (≥ 3.0), `numpy`. Se ejecuta en Colab con GPU.

## 1. `Conv1D` sobre una secuencia

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)
T, F = 64, 1
conv = keras.Sequential([
    keras.Input(shape=(T, F)),
    layers.Conv1D(32, kernel_size=5, activation="relu"),   # padding='valid' por defecto
])
print("salida Conv1D (valid):", conv.output_shape)   # (None, 60, 32)

## 2. `padding="causal"`: el output en t no ve el futuro

In [ ]:
causal = keras.Sequential([
    keras.Input(shape=(T, F)),
    layers.Conv1D(32, kernel_size=5, padding="causal", activation="relu"),
])
# padding causal: output[t] depende SOLO de input[<= t] y conserva la longitud
print("salida causal:", causal.output_shape)   # (None, 64, 32)

## 3. `dilation_rate` y receptive field

In [ ]:
def receptive_field(kernel, dilations):
    rf = 1
    for d in dilations:
        rf += (kernel - 1) * d
    return rf

capa_dil = layers.Conv1D(16, 2, padding="causal", dilation_rate=4)  # ve t, t-4, t-8...
print("dilation_rate:", capa_dil.dilation_rate)
print("receptive field (kernel=2, dils=[1,2,4,8]):", receptive_field(2, [1, 2, 4, 8]))

## 4. Mini-WaveNet: stack de convoluciones causales dilatadas

In [ ]:
entrada = keras.Input(shape=(T, F))
z = entrada
for tasa in (1, 2, 4, 8, 16, 32):        # dilations exponenciales
    z = layers.Conv1D(32, 2, padding="causal",
                      dilation_rate=tasa, activation="relu")(z)
z = layers.Conv1D(1, 1)(z)               # proyección 1x1 final
wavenet = keras.Model(entrada, z)
wavenet.compile(optimizer="adam", loss="mae")
print("receptive field total:", receptive_field(2, [1, 2, 4, 8, 16, 32]))
wavenet.summary()

## 5. Conv1D vs LSTM para forecasting

In [ ]:
conv_fore = keras.Sequential([
    keras.Input(shape=(T, F)),
    layers.Conv1D(32, 5, padding="causal", activation="relu"),
    layers.Conv1D(32, 5, padding="causal", activation="relu"),
    layers.GlobalAveragePooling1D(),     # mejor que Flatten: muchos menos params
    layers.Dense(1),
])
conv_fore.compile(optimizer="adam", loss="mae")
print("params Conv1D:", conv_fore.count_params())
# Conv1D no tiene dependencia temporal secuencial -> mucho más rápido en GPU que LSTM.

## 6. Cómo crece el receptive field con dilations exponenciales

In [ ]:
for capas in range(1, 7):
    rf = receptive_field(2, [2 ** i for i in range(capas)])
    print(f"{capas} capas dilatadas -> receptive field = {rf} pasos")
# Con N capas y dilations 1,2,...,2^(N-1) el rango es ~2^N: crecimiento exponencial.

## Ejercicios

1. **Conv1D vs LSTM**: entrená ambos para forecasting y compará MAE y tiempo por época.
2. **Dilations**: armá un stack `dilation_rate ∈ {1,2,4,8}` y calculá el receptive field.
3. **Mini-WaveNet**: extendé el stack a rates `{1,...,512}` para una serie larga.
4. **GlobalAvgPool**: reemplazá `Flatten → Dense` por `GlobalAveragePooling1D` y compará params.

## Conclusiones

- `Conv1D` procesa todas las posiciones en **paralelo** → 5-20× más rápido que LSTM en GPU.
- `padding="causal"` evita ver el futuro; es imprescindible en forecasting.
- `dilation_rate` amplía el **receptive field** sin agregar capas ni parámetros.
- **WaveNet** apila convoluciones causales dilatadas (1,2,4,...,512).
- Usá `GlobalAveragePooling1D` antes del `Dense` final para no explotar el nº de parámetros.